In [8]:
import pandas as pd
import numpy as np
import os
# import standardscaler
from sklearn.preprocessing import StandardScaler
import joblib
from tqdm import tqdm


# Functions

In [9]:
def fill_na_based_on_day(df):
    for col in df.columns:
        for i in range(1, len(df)):
            # Get the day of the week (Monday=0, Sunday=6)
            day_of_week = df.index[i].weekday()
            if pd.isna(df.iloc[i][col]):
                # For Tuesday (1) to Friday (4), and Sunday (6), replace NaN with the previous day's value
                if day_of_week in [1, 2, 3, 4, 6]:
                    if i >= 24:
                        df.iloc[i, df.columns.get_loc(col)] = df.iloc[i - 24, df.columns.get_loc(col)]
                # For Monday (0) and Saturday (5), replace NaN with the value from one week ago
                elif day_of_week in [0, 5]:
                    if i >= 24*7:
                        df.iloc[i, df.columns.get_loc(col)] = df.iloc[i - 24*7, df.columns.get_loc(col)]
    return df

def get_datasets(country):
    generation = pd.read_csv(os.path.join("processed_data", country, "generation.csv"), index_col=0)
    generation.index = pd.to_datetime(generation.index)

    da_price = pd.read_csv(os.path.join("processed_data", country, "da_price.csv"), index_col=0)
    da_price.index = pd.to_datetime(da_price.index)

    crossborder = pd.read_csv(os.path.join("processed_data", country, "crossborder.csv"), index_col=0)
    crossborder.index = pd.to_datetime(crossborder.index)
    crossborder = crossborder.fillna(0)
    
    demand = pd.read_csv(os.path.join("processed_data", country, "load.csv"), index_col=0)
    demand.index = pd.to_datetime(demand.index)
    
    ws_forecast = pd.read_csv(os.path.join("processed_data", country, "ws_forecast.csv"), index_col=0)
    ws_forecast.index = pd.to_datetime(ws_forecast.index)
    

    def process_df(df):
        df = df.loc[:"2024-12-31 23:00:00"].copy()
        df.index = pd.to_datetime(df.index)
        df = fill_na_based_on_day(df)
        df = df.iloc[24:].copy()
        df = df.dropna(axis=1, thresh=0.9*len(df))
        df = df.fillna(method='ffill')
        return df
        
    generation = process_df(generation)
    generation = generation.rename(columns={col:col.replace(" - Actual Aggregated [MW]", "generation").strip() for col in generation.columns if " - Actual Aggregated [MW]" in col})
    da_price = process_df(da_price)
    crossborder = process_df(crossborder)
    demand = process_df(demand)
    ws_forecast = process_df(ws_forecast)
    
    os.makedirs(os.path.join("cut_data",country), exist_ok=True)
    return [generation, da_price, crossborder, demand, ws_forecast]

In [10]:
def get_time_features(X):

    def create_features(df, label=None):
        """
        Creates time series features from datetime index.
        """
        df = df.copy()
        df['date'] = df.index
        df['dayofweek'] = df['date'].dt.dayofweek
        df['month'] = df['date'].dt.month
        df['dayofyear'] = df['date'].dt.dayofyear
        df['dayofmonth'] = df['date'].dt.day
        df['date_offset'] = (df.date.dt.month*100 + df.date.dt.day - 320)%1300

        X = df[['dayofweek','month',
               'dayofyear','dayofmonth',]]
        if label:
            y = df[label]
            return X, y
        return X

    X = create_features(X)


    def cyclic_transform(values, cardinality):
        arr1 = np.sin(2*np.pi*values/cardinality)
        arr2 = np.cos(2*np.pi*values/cardinality)

        return arr1, arr2


    def cyclic_column(df, col, cadinality):
        arr1, arr2 = cyclic_transform(df[col], cadinality)
        df[col + '_sin'] = arr1
        df[col + '_cos'] = arr2
        df.drop(col, axis = 1, inplace = True)
        return df

    for col in ['dayofweek', 'month', 'dayofyear', 'dayofmonth']:
        X = cyclic_column(X, col, X[col].max())
    
    return X

In [ ]:
def scale_df(df):
    scaler = StandardScaler()
    scaler.fit(df.iloc[:int(len(df)*0.4)])
    df = pd.DataFrame(scaler.transform(df), columns=df.columns, index=df.index)
    return df, scaler

In [12]:
def input_output(generation, da_price, crossborder, demand, ws_forecast, time):
    inputs = []
    outputs = []
    indexes = []
   
    for i in range(len(da_price)//24-7):
        # Generation
        #7 day lag
        gen7 = generation.iloc[i*24:(i+1)*24].values.flatten()
        # 3 day lag
        gen3 = generation.iloc[(i+4)*24:(i+5)*24].values.flatten()
        # 2 day lag
        gen2 = generation.iloc[(i+5)*24:(i+6)*24].values.flatten()
        # 1 day 0-7 lag
        gen1 = generation.iloc[(i+6)*24:(i+6)*24+7].values.flatten()
        
        gen_final = np.concatenate([gen7, gen3, gen2, gen1])
        
        # Cross Border
        #7 day lag
        cross7 = crossborder.iloc[i*24:(i+1)*24].values.flatten()
        #3 day lag
        cross3 = crossborder.iloc[(i+4)*24:(i+5)*24].values.flatten()
        #3 day lag
        cross2 = crossborder.iloc[(i+5)*24:(i+6)*24].values.flatten()
        #1 day 0-7 lag
        cross1 = crossborder.iloc[(i+6)*24:(i+6)*24+7].values.flatten()
 
        cross_final = np.concatenate([cross7, cross3, cross2, cross1])
        
        # DA Price
        # 7 day lag
        daprice_7 = da_price.iloc[i*24:(i+1)*24].values.flatten()
  
        # 3 day lag
        daprice_3 = da_price.iloc[(i+4)*24:(i+5)*24].values.flatten()
  
        # 2 day lag
        daprice_2 = da_price.iloc[(i+5)*24:(i+6)*24].values.flatten()
  
        # 1 day lag
        daprice_1 = da_price.iloc[(i+6)*24:(i+7)*24].values.flatten()
  
        daprice_final = np.concatenate([daprice_7, daprice_3, daprice_2, daprice_1])
        
        # Demand
        # 0 day lag
        demand_final = demand.iloc[(i+7)*24:(i+8)*24].values.flatten()
        
        
        # WS Forecast
        # 0 day lag
        ws_final = ws_forecast.iloc[(i+7)*24:(i+8)*24].values.flatten()
      
      
        # Time features
        # 0 day lag
        time_final = time.iloc[(i+7)*24].values.flatten()
        
        
        # input data
        input_final = np.concatenate([gen_final, cross_final, daprice_final, demand_final, ws_final, time_final])
        inputs.append(list(input_final))
        
        # output data
        daprice_0 = da_price.iloc[(i+7)*24:(i+8)*24].values.flatten()
        outputs.append(list(daprice_0))
        
        indexes.append(da_price.index[(i+7)*24])

      
      
    gen_colnames = []
    for i in [7,3,2,1]:
        if i!=1:
            for j in range(24):
                for col in generation.columns:
                    gen_colnames.append(f"{col} lag{i}_hour{j}")
        else:
            for j in range(7):
                for col in generation.columns:
                    gen_colnames.append(f"{col} lag{i}_hour{j}")
                    
    cross_colnames = []
    for i in [7,3,2,1]:
        if i!=1:
            for j in range(24):
                for col in crossborder.columns:
                    cross_colnames.append(f"{col} lag{i}_hour{j}")
        else:
            for j in range(7):
                for col in crossborder.columns:
                    cross_colnames.append(f"{col} lag{i}_hour{j}")
                    
    daprice_colnames = []
    for i in [7,3,2,1]:
        for j in range(24):
            for col in da_price.columns:
                daprice_colnames.append(f"{col} lag{i}_hour{j}")
                
    demand_colnames = []
    for j in range(24):
        for col in demand.columns:
            demand_colnames.append(f"{col} hour{j}")
            
    ws_colnames = []
    for j in range(24):
        for col in ws_forecast.columns:
            ws_colnames.append(f"{col} hour{j}")
    
    time_colnames = list(time.columns)
    
    pred_colnames = [str(i) for i in range(24)]
    
    colnames_final = gen_colnames + cross_colnames + daprice_colnames + demand_colnames + ws_colnames + time_colnames
    
    inputs = pd.DataFrame(inputs, columns=colnames_final, index=indexes)
    outputs = pd.DataFrame(outputs, columns=pred_colnames, index=indexes)
    
    return inputs, outputs

In [ ]:
def save_data(inputs,outputs,country):
    
    inputs.to_csv(os.path.join("cut_data",country,"inputs.csv"))
    outputs.to_csv(os.path.join("cut_data",country,"outputs.csv"))
    
    return f'{country} is done'

# Processing

In [ ]:
country_list = ["Estonia", "Finland", "Hungary", "Latvia", "Poland"]
for country in tqdm(country_list):
    # Get datasets
    generation, da_price, crossborder, demand, ws_forecast = get_datasets(country)
    time = get_time_features(da_price)
    
    # Create input and output datasets
    inputs, outputs = input_output(generation, da_price, crossborder, demand, ws_forecast, time)
    
    save_data(inputs,outputs,country)
    
    #print(f'{country} is done')

100%|██████████| 5/5 [31:54<00:00, 382.94s/it]
